# Introduction to Neuroimaging Data Analysis with Python

> **Repository:** [https://github.com/ArunimGuchait/neuroimaging-intro](https://github.com/ArunimGuchait/neuroimaging-intro)  
> **Open in Colab:** [Launch this notebook](https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb)

This notebook is a **beginner-friendly guide** to analyzing brain imaging data in Python. No prior knowledge of neuroimaging or advanced programming is assumed.

---

## What is Neuroimaging?

**Neuroimaging** is the use of various techniques to image the structure and function of the brain. Unlike a single photograph, brain images are typically **3D volumes** (think of a loaf of bread made of thin slices), and for functional studies we add **time**, giving us **4D data** (3 space dimensions + 1 time dimension).

## What is fMRI?

**Functional Magnetic Resonance Imaging (fMRI)** measures brain *activity* indirectly by detecting changes in blood flow and blood oxygenation. When a brain region becomes more active, it consumes more oxygen, and the body responds by sending more oxygenated blood to that area. fMRI detects this **BOLD** (Blood Oxygen Level Dependent) signal. So we do not see "thoughts" directly—we see where the brain is working harder over time.

## What You Will Learn Here

1. **Concepts**: Voxels, NIfTI files, 4D data, and the Python tools used in neuroimaging.
2. **Data**: How to download and load publicly available fMRI datasets.
3. **Inspection**: How to explore the shape, size, and content of imaging data.
4. **Visualization**: How to plot brain slices and time series.
5. **First steps in analysis**: Extracting signals from regions and a brief look at preprocessing.

## How to use this notebook

- Read the text, then run the code cell below it.
- If you get errors, read the last line first; it usually explains the problem.
- Running cells in order matters because later cells use variables created earlier.

Let's get started.

---
## 1. Environment setup

To run this notebook, set up a Python environment with the neuroimaging libraries installed. Choose one of the options below.

*For a detailed explanation of virtual environments and the setup process, see the [Introduction to Python Programming for Neuroimaging Data Analysis](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb) companion notebook.*

**Quick note:** If you already created `neuro-env` in Chapter 01, you can reuse it here and just install the requirements.

**Option A – Install into the current kernel (local Jupyter / conda):**  
Run the cell below to install the requirements into whatever kernel you have selected. Keep the notebook's working directory as the folder that contains `requirements.txt`.

**Option B – Google Colab:**  
If you opened this notebook in [Google Colab](https://colab.research.google.com/github/ArunimGuchait/neuroimaging-intro/blob/main/introduction_neuroimaging_analysis.ipynb), run the **next cell** (it will detect Colab and install the required packages automatically). No local install needed.

**Option C – Create a virtual environment with pip (recommended for local use):**  
In a terminal, from the folder containing `requirements.txt`:
```bash
# Create virtual environment
python -m venv neuro-env          # or: python3 -m venv neuro-env on Linux/macOS

# Activate it
neuro-env\Scripts\activate        # Windows
source neuro-env/bin/activate     # Linux/macOS

# Install packages
pip install -r requirements.txt
```

**Option D – Install packages directly (one line):**
```bash
pip install numpy pandas matplotlib scipy nibabel nilearn jupyter
```

Then start Jupyter:
```bash
jupyter notebook introduction_neuroimaging_analysis.ipynb
```

**What each library does:**
- **nibabel**: Reads and writes neuroimaging file formats (e.g. NIfTI).
- **nilearn**: Analyzing and plotting brain images; downloading public fMRI datasets.
- **numpy**: Numerical arrays (brain images are multi-dimensional arrays).
- **pandas**: Tables for subject info and metadata.
- **matplotlib & scipy**: Plotting and statistical functions.

In [ ]:
# Install requirements: works both locally and on Google Colab.
# Run this cell once per session.

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Google Colab: install packages (no requirements.txt needed)
    print("Google Colab detected. Installing packages...")
    get_ipython().system(
        "pip install --quiet numpy pandas matplotlib scipy nibabel nilearn"
    )
    print("Done. Optional: mount Google Drive and set NILEARN_DATA to a Drive folder to cache data across sessions.")
else:
    # Local (Jupyter / conda): use requirements.txt
    # Ensure your working directory is the folder containing this notebook and requirements.txt.
    get_ipython().run_line_magic("pip", "install -r requirements.txt")

**Optional (Google Colab only):** Run the cell below to mount Google Drive and set the Nilearn data directory to a folder on Drive. Then the downloaded dataset will be cached on your Drive and you won’t need to re-download it when the Colab runtime restarts. Skip this if you’re running locally or don’t need to persist the cache.

In [ ]:
# Optional: use only on Google Colab if you want to cache data on Drive.
try:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    nilearn_data_dir = "/content/drive/MyDrive/nilearn_data"
    os.makedirs(nilearn_data_dir, exist_ok=True)
    os.environ["NILEARN_DATA"] = nilearn_data_dir
    print("Nilearn data will be cached in:", nilearn_data_dir)
except ImportError:
    print("Not in Colab; skipping Drive mount.")

In [ ]:
# Check that key packages are available (run this cell first)
import sys
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import nibabel as nib
    import nilearn
    print("Python:", sys.version)
    print("NumPy:", np.__version__)
    print("NiBabel:", nib.__version__)
    print("Nilearn:", nilearn.__version__)
    print("\nAll required packages are installed. You can proceed.")
except ImportError as e:
    print("Missing package:", e)
    print("Please install with: pip install -r requirements.txt")

---
## 2. Neuroimaging data basics: NIfTI and voxels

### NIfTI format

Most research brain imaging data is stored in **NIfTI** format (file extension `.nii` or `.nii.gz`). A NIfTI file contains:

1. **A 3D or 4D array of numbers**
   - Each number represents the signal at one small box in the brain at one time point.
2. **An "affine" matrix**
   - This describes how array indices map to real-world coordinates (e.g. millimetres in "MNI" space, a standard brain template). It encodes position, orientation, and voxel size.

So when we "load" a NIfTI file in Python, we get both the array of numbers and the information needed to interpret them in physical space.

### Voxels

The 3D image is divided into a grid of tiny cubes called **voxels** (volume elements). Each voxel has one number per time point (in 4D fMRI). So:

- **3D (structural)**: one number per voxel -> e.g. anatomy (T1-weighted image).
- **4D (functional)**: many time points per voxel -> a **time series** of BOLD signal at that location.

Typical fMRI resolution might be 3 mm x 3 mm x 3 mm voxels and hundreds of time points (e.g. one every 1-2 seconds). So the "shape" of a 4D fMRI file is often something like `(64, 64, 30, 200)` meaning 64 x 64 x 30 voxels in space and 200 time points.

Beginner summary:
- **Voxel** = 3D pixel (a tiny cube).
- **Volume** = one 3D brain image.
- **Time series** = a line of values at one voxel across time.

---
## 3. Downloading publicly available fMRI data

We will use **Nilearn's built-in dataset fetchers** to download a small amount of real, publicly available data. No need to create an account or manually find URLs.

We use the **development fMRI dataset** (OpenNeuro dataset **ds000228**): movie-watching fMRI from children and adults, downsampled to 4 mm for teaching. It includes:

- **func**: 4D NIfTI files (preprocessed functional scans).
- **confounds**: TSV files with motion and other nuisance variables (used in real analyses to "clean" the signal).
- **phenotypic**: Subject information (e.g. age, sex).

We will fetch only **2 subjects** to keep download size and run time small. The first time you run this, it will download data to a folder (usually `~/nilearn_data` or similar); later runs will reuse the cache.

What to expect:
- First run = download (may take a few minutes).
- Later runs = quick (uses cached files).
- You will see file paths printed so you know where the data lives.

In [ ]:
from nilearn import datasets
import os

# Fetch development fMRI data: 2 subjects, both age groups, with simplified confounds
# First run may take a few minutes while data downloads.
# If you set NILEARN_DATA (e.g. in the optional Colab + Drive cell), data is cached there.
base_data_dir = os.environ.get(
    "NILEARN_DATA",
    os.path.join(os.path.expanduser("~"), "nilearn_data"),
)
data_dir = os.path.join(base_data_dir, "development_fmri")
print("Data will be stored in:", data_dir)
print("Downloading (or loading from cache)...")

development_data = datasets.fetch_development_fmri(
    n_subjects=2,
    age_group="both",
    reduce_confounds=True,
    data_dir=base_data_dir,
)

# What did we get?
print("\nKeys in the dataset:", list(development_data.keys()))
print("\nFunctional files (4D NIfTI per subject):", development_data.func)
print("\nConfound files (TSV per subject):", development_data.confounds)
print("\nPhenotypic (subject info):")
print(development_data.phenotypic)

---
## 4. Loading and inspecting the data with NiBabel

We use **NiBabel** to load a NIfTI file. The result is a "image" object. We do not load the whole array into memory until we call `.get_fdata()`. For very large files, you can work with nibabel's memory-mapped arrays to save RAM.

- **img.shape**: dimensions (e.g. x, y, z, time).
- **img.affine**: the 4x4 matrix that maps voxel indices to real-world coordinates (e.g. mm in MNI space).
- **img.get_fdata()**: returns a NumPy array of the image data (float). The last dimension is usually time for 4D fMRI.

Plain-language steps:
1. Pick a file path.
2. Load it with NiBabel (gives an image object).
3. Ask for the data array and inspect its shape.

In [ ]:
import nibabel as nib
import numpy as np

# Pick the first subject's functional scan
first_subject_func = development_data.func[0]
print("File path:", first_subject_func)

# Load the NIfTI file (this does not load the full array into memory yet)
img = nib.load(first_subject_func)
print("\nImage type:", type(img))
print("Image shape (x, y, z, time):", img.shape)
print("Affine (voxel-to-world mapping):\n", img.affine)

# Load the actual array of numbers into memory
data = img.get_fdata()
print("\nData array shape:", data.shape)
print("Data type:", data.dtype)
print("Min / Max value:", data.min(), "/", data.max())
print("Mean value (over all voxels and time):", np.mean(data))

### Slicing the 4D array

The array is indexed as `[x, y, z, time]` (or similarly depending on the file). So:

- `data[:, :, :, 0]` -> 3D volume at the first time point.
- `data[32, 32, 15, :]` -> time series at a single voxel (x=32, y=32, z=15).

We can also compute a **mean volume** across time to get a static "average" brain for this subject, which is useful for visualization.

Beginner tip: think of the 4D file as a **stack of 3D volumes**, one per time point.

In [ ]:
# Slicing examples
n_x, n_y, n_z, n_time = data.shape
print("Dimensions: x={}, y={}, z={}, time={}".format(n_x, n_y, n_z, n_time))

# First time point (3D volume)
first_volume = data[:, :, :, 0]
print("\nFirst time point shape:", first_volume.shape)

# Time series at one voxel (middle of the brain, approximate)
center_x, center_y, center_z = n_x // 2, n_y // 2, n_z // 2
time_series_at_voxel = data[center_x, center_y, center_z, :]
print("Time series length:", len(time_series_at_voxel))

# Mean volume across time (often used for visualization)
mean_volume = np.mean(data, axis=-1)
print("Mean volume shape:", mean_volume.shape)

---
## 5. Visualizing brain images with Nilearn

Nilearn provides plotting functions that know about brain geometry: they show slices in **anatomical directions** (axial = horizontal, sagittal = side, coronal = front/back) and handle the affine so that orientation is correct.

- **plot_epi**: for "EPI" (echo-planar) functional images. Useful for raw or mean functional data.
- **plot_stat_map**: for statistical maps (e.g. t-maps, contrast maps); can overlay on a background and use a colour scale.

We will plot the **mean functional image** (average across time) so we see the brain structure as captured by this fMRI scan.

Beginner note: these plots are for understanding and checking data quality, not for final scientific results.

In [ ]:
from nilearn import plotting

# Create a temporary NIfTI image for the mean volume (nilearn needs a NIfTI image, not raw array)
mean_img = nib.Nifti1Image(mean_volume, img.affine)

# Plot the mean EPI in three views (slices)
plotting.plot_epi(
    mean_img,
    title="Mean functional image (subject 1)",
    display_mode="mosaic",  # multiple slices in one figure
    colorbar=True,
)
plt.show()

### Plotting a single time point

Below we plot one 3D volume (e.g. the 50th time point) to see how a single "frame" of the fMRI looks. The signal is noisy; that is why we often average across time or smooth in space/time.

Beginner note: a single time point is rarely interpretable on its own. It is mainly useful for sanity checks.

In [ ]:
# One time point as a NIfTI image (3D)
time_point = 50
single_volume = data[:, :, :, time_point]
single_img = nib.Nifti1Image(single_volume, img.affine)

plotting.plot_epi(
    single_img,
    title=f"Single time point (t={time_point})",
    display_mode="mosaic",
    colorbar=True,
)
plt.show()

### Plotting a time series at one voxel

The BOLD signal changes over time. At a single voxel we get a 1D curve: intensity vs. time (in "TRs", i.e. repetition times or volume acquisitions). This is the **time series** we use for connectivity or task analyses. Here we plot the time series at the centre voxel we extracted earlier.

Beginner note: the y-axis is in **arbitrary units**; it is the relative change over time that matters.

In [ ]:
# Repetition time (TR) for this dataset: 2 seconds per volume (from dataset description)
TR = 2.0  # seconds
time_axis = np.arange(n_time) * TR

plt.figure(figsize=(10, 3))
plt.plot(time_axis, time_series_at_voxel, color="steelblue", linewidth=0.8)
plt.xlabel("Time (seconds)")
plt.ylabel("BOLD signal (arbitrary units)")
plt.title("Time series at one voxel (centre of the brain)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Confounds: what they are and why they matter

The BOLD signal is not only driven by neural activity. **Confounds** (nuisance variables) include:

- **Motion**: Head movement in the scanner shifts the brain in the image and causes large signal changes.
- **Physiological noise**: Heartbeat and breathing cause small, rhythmic changes in blood flow.
- **Scanner drift**: Slow changes in signal over the session.

In a full analysis pipeline we would **regress out** these confounds (e.g. using the columns in the confounds TSV) before estimating brain activity or connectivity. Here we only **load and inspect** the confounds file so you see what such a table looks like.

Beginner note: you do not need to memorize these columns; you just need to recognize that they exist and are used to clean data.

In [ ]:
# Load confounds for the first subject (TSV = tab-separated values)
confounds_path = development_data.confounds[0]
confounds_df = pd.read_csv(confounds_path, sep="\t")
print("Confounds shape:", confounds_df.shape)
print("Columns (first 15):", list(confounds_df.columns[:15]))
print("\nFirst 3 rows (first 5 columns):")
print(confounds_df.iloc[:3, :5])

---
## 7. Regions of interest (ROIs) and atlases

Often we are interested in the signal **averaged over a brain region** (e.g. "left motor cortex") rather than a single voxel. Such a region is called a **region of interest (ROI)**. An **atlas** is a labelled map of the brain: each voxel is assigned to a region (e.g. by a number or a name).

Nilearn can **fetch atlases** and **extract the mean time series** from each region. Here we use a small atlas (e.g. **Harvard-Oxford cortical atlas** or **AAL**) to get a few ROIs and plot their mean time series. This is a first step toward **connectivity** analysis (e.g. correlations between regions).

Beginner note: a ROI is just a group of voxels treated as one unit.

In [ ]:
from nilearn.maskers import NiftiLabelsMasker
from nilearn import datasets as ds

# Fetch a parcellation atlas (many atlases available; we use one with a small number of ROIs)
atlas = ds.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
print("Atlas keys:", atlas.keys())
print("Atlas file:", atlas.maps)

# NiftiLabelsMasker: for each label in the atlas, extract the mean time series
masker = NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize="zscore",  # z-score each ROI time series (mean=0, std=1)
    memory="nilearn_cache",
    verbose=0,
)

# Fit to our first subject's data and extract ROI time series
roi_time_series = masker.fit_transform(first_subject_func)
print("\nROI time series shape: (n_timepoints, n_regions) =", roi_time_series.shape)

### Plotting a few ROI time series

We plot the first 3 regions' time series. In a full analysis you would then compute correlations between regions (functional connectivity) or relate them to your task design.

Beginner note: these time series are **z-scored**, which means each one is scaled to have mean 0 and standard deviation 1.

In [ ]:
# Plot first 3 ROIs
n_rois_to_plot = 3
fig, axes = plt.subplots(n_rois_to_plot, 1, figsize=(10, 6), sharex=True)
time_axis_roi = np.arange(roi_time_series.shape[0]) * TR

for i in range(n_rois_to_plot):
    axes[i].plot(time_axis_roi, roi_time_series[:, i], linewidth=0.8)
    axes[i].set_ylabel(f"ROI {i+1} (z)")
    axes[i].grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (seconds)")
fig.suptitle("Mean time series for first 3 atlas regions (z-scored)")
plt.tight_layout()
plt.show()

### A first analysis: functional connectivity (correlation matrix)

**Functional connectivity** is the statistical association between time series from different brain regions (e.g. correlation). If two regions' BOLD signals go up and down together, they are often said to be "functionally connected." Below we compute the **correlation matrix** between all ROI time series and plot it as a heatmap. This is a simplified version of what connectivity studies do with many subjects and statistical testing.

Beginner note: correlation values range from **-1** (opposite) to **+1** (together). Values near 0 mean little relationship.

In [ ]:
# Correlation matrix between ROI time series (columns of roi_time_series)
# Each column is one ROI; we correlate columns with each other.
correlation_matrix = np.corrcoef(roi_time_series.T)

# Plot as heatmap (show a subset of ROIs if there are many)
n_show = min(12, correlation_matrix.shape[0])
plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix[:n_show, :n_show], cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xlabel("ROI index")
plt.ylabel("ROI index")
plt.title("Functional connectivity (correlation between ROI time series)")
plt.tight_layout()
plt.show()

---
## 8. Smoothing (conceptual and practical)

**Spatial smoothing** means averaging the signal over nearby voxels (often with a Gaussian kernel). It:

- **Increases signal-to-noise ratio** by averaging out random noise.
- **Makes data more normally distributed**, which helps many statistical methods.
- **Reduces effective resolution** (details are blurred). Typical smoothing is 1-2x the voxel size (e.g. 6 mm FWHM for 3 mm voxels).

Nilearn's **image** module provides `smooth_img()`. Here we smooth the mean volume and compare it to the unsmoothed mean. In a full pipeline, smoothing is usually applied to the 4D data before statistical analysis.

Beginner note: smoothing is a tradeoff between clarity and detail.

In [ ]:
from nilearn import image

# Smooth the mean image with a 6 mm FWHM Gaussian kernel
# (smoothing 4D data would use the same function on the 4D NIfTI)
mean_img_nifti = nib.Nifti1Image(mean_volume, img.affine)
smoothed_mean = image.smooth_img(mean_img_nifti, fwhm=6)

# Visual comparison: one slice (middle z)
z_slice = n_z // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(mean_volume[:, :, z_slice].T, origin="lower", cmap="gray")
axes[0].set_title("Mean volume (no smoothing)")
axes[0].axis("off")
axes[1].imshow(smoothed_mean.get_fdata()[:, :, z_slice].T, origin="lower", cmap="gray")
axes[1].set_title("Mean volume (6 mm FWHM smoothed)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

---
## 9. Summary and next steps

You have seen:

- **Concepts**: Neuroimaging and fMRI (BOLD), NIfTI files, voxels, 3D/4D data, affines, confounds.
- **Libraries**: NiBabel (load/save NIfTI), Nilearn (datasets, masking, smoothing, plotting), NumPy and Pandas.
- **Data**: Downloading public development fMRI data and inspecting its shape and confounds.
- **Visualization**: Mean EPI, single time point, voxel time series, and ROI time series.
- **First steps**: ROI extraction with an atlas and spatial smoothing.

### What typically comes next

1. **Preprocessing**: Motion correction, slice-time correction, coregistration to anatomy, normalization to a standard space (e.g. MNI), and confound regression. Pipelines like **fMRIPrep** automate much of this.
2. **First-level analysis**: Fit a model (e.g. General Linear Model, GLM) to each subject's 4D data using a task design (e.g. condition onsets and durations) to get contrast maps (e.g. "task A vs baseline").
3. **Second-level (group) analysis**: Combine subjects (e.g. one-sample t-test on contrast images) to make inferences about the population.
4. **Connectivity**: Use ROI time series (or whole-brain parcellations) to compute correlation matrices, graph metrics, or dynamic connectivity.

### If you want to keep learning

- Re-run this notebook with more subjects (`n_subjects=10`).
- Try a different atlas or dataset in Nilearn.
- Focus on one step (download, inspect, plot) and repeat until it feels familiar.
- Check out [Chapter 01 (Python Prequel)](https://github.com/ArunimGuchait/neuroimaging-intro/blob/main/introduction_python_for_neuroimaging.ipynb) if you want to deepen your Python fundamentals.

### Further resources

- **Nilearn**: [User guide](https://nilearn.github.io/stable/user_guide.html) and [examples](https://nilearn.github.io/stable/auto_examples/index.html).
- **NiBabel**: [Documentation](https://nipy.org/nibabel/).
- **OpenNeuro**: [openneuro.org](https://openneuro.org) for public neuroimaging datasets.
- **BIDS**: [bids.neuroimaging.io](https://bids.neuroimaging.io) for organizing and sharing data in a standard way.
- **GitHub Repository**: [neuroimaging-intro](https://github.com/ArunimGuchait/neuroimaging-intro) for issues and updates.

You can re-run this notebook with more subjects (`n_subjects=10`) or try other Nilearn datasets (e.g. `fetch_neurovault()`, `fetch_openneuro_dataset()`) to explore further.